# 🏆 Churn Prediction: Master Production Pipeline (Visual & Recall Optimized)
**Core Priority: RECALL** - Catching every potential churner.
**Status**: FINAL AUDIT - Syntax verified.

In [ ]:
# 1. Hardware & Environment Check
!nvidia-smi
!pip install opendatasets mlflow xgboost shap python-dotenv seaborn dagshub -q

import torch
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    print(f"✅ CUDA Detected: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU found. Training will run on CPU.")

import opendatasets as od
import os, json, pandas as pd, numpy as np, mlflow, mlflow.sklearn, matplotlib.pyplot as plt, seaborn as sns, shap, dagshub
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix, roc_curve, auc
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from mlflow.models import infer_signature

In [ ]:
# 2. Auth & Data
dagshub.init(repo_owner="nhannhb92", repo_name="msa24-ddm501-group6-final-project", mlflow=True)

def clean_dataset(df):
    df.columns = [col.lower().replace(' ', '_') for col in df.columns]
    df = df.drop(columns=[c for c in ['customerid', 'last_interaction'] if c in df.columns])
    if 'age' in df.columns: df = df[(df['age'] >= 18) & (df['age'] <= 120)]
    # FIXED: Corrected list comprehension check
    for col in [c for c in ['total_spend', 'totalcharges'] if c in df.columns]:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df.dropna(subset=['churn'])

if not os.path.exists("customer-churn-dataset"):
    od.download("https://www.kaggle.com/datasets/muhammadshahidazeem/customer-churn-dataset")

df_tr_raw = clean_dataset(pd.read_csv("customer-churn-dataset/customer_churn_dataset-training-master.csv"))
df_test = clean_dataset(pd.read_csv("customer-churn-dataset/customer_churn_dataset-testing-master.csv"))
df_train, df_val = train_test_split(df_tr_raw, test_size=0.2, random_state=42, stratify=df_tr_raw['churn'])
print("✅ 3-Fold Data Ready.")

## 3. Deep Exploratory Data Analysis & Integrity Audit

In [ ]:
plt.figure(figsize=(20, 5))

# 3.1 Churn Imbalance Plot
plt.subplot(1, 3, 1)
sns.countplot(x='churn', data=df_train, palette='magma')
plt.title("Churn Imbalance (Training Set)")

# 3.2 Feature Correlation (Numerical)
plt.subplot(1, 3, 2)
corr = df_train.select_dtypes(include=[np.number]).corr()['churn'].sort_values().drop('churn')
corr.plot(kind='barh', color='darkcyan')
plt.title("Correlation with Churn (Numerical)")

# 3.3 Data Distribution Audit (Train vs Test)
audit_df = pd.DataFrame({
    'Dataset': ['Train']*len(df_train) + ['Test']*len(df_test),
    'Churn': list(df_train['churn']) + list(df_test['churn'])
})
plt.subplot(1, 3, 3)
sns.barplot(x='Dataset', y='Churn', data=audit_df, palette='Set2')
plt.title("Integrity Check: Churn Rate (Train vs Test)")

plt.tight_layout(); plt.show()

print(f"Training Churn Rate: {df_train.churn.mean()*100:.2f}%")
print(f"Testing Churn Rate: {df_test.churn.mean()*100:.2f}%")

## 4. Recall-Optimized Master Training Cycle

In [ ]:
def run_recall_experiment(df_tr, df_va, df_te, model_type):
    X_tr, y_tr = df_tr.drop(columns=['churn']), df_tr['churn']
    X_va, y_va = df_va.drop(columns=['churn']), df_va['churn']
    X_te, y_te = df_te.drop(columns=['churn']), df_te['churn']
    
    preprocessor = ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), X_tr.select_dtypes(include=[np.number]).columns.tolist()),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), X_tr.select_dtypes(include=['object']).columns.tolist())
    ])
    
    if model_type == "xgboost":
        pw = (y_tr == 0).sum() / (y_tr == 1).sum()
        clf = XGBClassifier(scale_pos_weight=pw, device='cuda' if HAS_GPU else 'cpu', tree_method='hist')
        params = {'clf__n_estimators': [100, 200]}
    elif model_type == "random_forest":
        clf = RandomForestClassifier(random_state=42)
        params = {'clf__n_estimators': [100, 200]}
    else:
        clf = LogisticRegression(class_weight='balanced', max_iter=2000)
        params = {'clf__C': [0.1, 1.0]}
        
    mlflow.set_experiment("Churn_Recall_Priority_FullEDA")
    with mlflow.start_run(run_name=f"NB_MASTER_{model_type.upper()}"):
        pipeline = Pipeline([('pre', preprocessor), ('clf', clf)])
        search = RandomizedSearchCV(pipeline, params, n_iter=1, cv=2, scoring='recall')
        search.fit(X_tr, y_tr)
        best_model = search.best_estimator_
        
        y_pred = best_model.predict(X_te)
        t_rec = recall_score(y_te, y_pred)
        mlflow.log_metrics({"test_recall": t_rec, "test_f1": f1_score(y_te, y_pred)})
        
        # Result Visual: ROC
        y_probs = best_model.predict_proba(X_te)[:, 1] if hasattr(best_model, 'predict_proba') else y_pred
        fpr, tpr, _ = roc_curve(y_te, y_probs)
        plt.figure(); plt.plot(fpr, tpr, label=f'AUC={auc(fpr,tpr):.2f}'); plt.title(f"ROC: {model_type}"); plt.legend(); plt.show()
        
        signature = infer_signature(X_tr, best_model.predict(X_tr))
        mlflow.sklearn.log_model(best_model, "model", registered_model_name=f"CustomerChurnModel_{model_type}", signature=signature)
        print(f"✅ {model_type.upper()} Registered - Recall: {t_rec:.4f})")

for m in ["xgboost", "logistic_regression", "random_forest"]:
    run_recall_experiment(df_train, df_val, df_test, m)